## End-to-end demo of CREST using MNIST data 

The entire notebook should run in < 60 seconds, and use ~600MB of memory.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Save the start time
import time
start_time = time.time()

# Allow crest to be imported
%cd -q ../..

#### Load MNIST

In [ ]:
import tensorflow as tf

# Load data and define constants
xy_train, xy_test = tf.keras.datasets.mnist.load_data()   
n_classes  = 10
patch_size = 28

#### Create the datasets

In [ ]:
from crest.src.data.loading import StructuredDataset

# Set input/output key labels
INP = 'image'
OUT = 'class'

# Define the train/test datasets, as well as the array labels
ds_train = StructuredDataset(*xy_train, labels=[INP, OUT])
ds_test  = StructuredDataset(*xy_test,  labels=[INP, OUT])

#### Create the batchers

In [ ]:
from crest.src.data import Batcher

# Note: workers=0 is faster for this, but may cause issues in a notebook
# since the background threads will run in the main process 
batch_kwargs = {
    'batch_size' : 200,            # 200 samples in each batch
    'features'   : [[INP], [OUT]], # [{'image': ...}, {'class': ...}]
    'repeat'     : True,           # Yield indefinitely
    'workers'    : 0,              # Number of background processes
}
batch_train = Batcher(ds_train, **batch_kwargs)
batch_kwargs['shuffle'] = False    #turn shuffle off for prediction
batch_test  = Batcher(ds_test,  **batch_kwargs)

#### Create the CREST graph

In [ ]:
from tensorflow.keras.layers import Conv2D, Dropout, Flatten, Dense, Lambda
from crest.src.model import HierarchalTensorGraph as HTG

def crest_layer(layer):
    """ Wrap a tensorflow layer in a HierarchalTensorGraph """
    graph = HTG(lambda d: {OUT: layer(d[INP])}, name=layer.name)
    #set inputs/outputs
    graph.inputs = {INP:None}
    graph.outputs = {OUT:None}
    return graph

# Define the tensorflow layers, and convert to HTG objects
layers = [crest_layer(layer) for layer in [
    Lambda(lambda x: tf.cast(x, tf.float32)),         # Cast dtype
    Lambda(lambda x: tf.expand_dims(x, axis=-1)),     # Add feature dimension
    Conv2D(10, 3, padding='same', activation='relu'), # 28x28x1 -> 28x28x10
    Flatten(),                                        # 28x28x10 -> 7840
    Dropout(0.3),                                     # 30% of data dropped
    Dense(n_classes, activation='softmax'),           # Class probabilities
]]

# Rename outputs to use as inputs (except the last layer)
for i in layers[:-1]:
    i.rename_io(outputs_map={OUT:INP})
    
# Create the CREST graph: ensure tensor shapes are specified for the inputs and outputs
graph = HTG(
    name='MNIST',
    inputs={INP : tf.TensorSpec(shape=[None,patch_size, patch_size])},
    outputs={OUT : tf.TensorSpec(shape=[None,10])}
)
print(list(map(graph.add_edge, ['input']+layers, layers+['output'])))

#### Build anad train CREST Keras model

In [ ]:
from crest import Model

build_kwargs= {
    'optimizer' : 'Adam', 
    'loss'      : 'sparse_categorical_crossentropy', 
    'metrics'   : 'sparse_categorical_accuracy'
}

model = Model(graph)
model.build(**build_kwargs)
model.fit(batch_train, **{
    'validation_data'  : batch_test,
    'steps_per_epoch'  : 300, # 60k samples / 200 samples per batch
    'validation_steps' : 50,  # 10k samples / 200 samples per batch
    'epochs'           : 5,
    'verbose'          : 1,
})


#### Evaluate the model

In [ ]:
import numpy as np

pred_kwargs = {
    'batch_size' : 200,
    'verbose'    : False
}

# Generate test predictions from a single batch
(x,y) = next(batch_test)
htg_pred = graph(x)                          # Using HTG
model_pred = model.predict(x,**pred_kwargs)  # Using predict
print('test diff batch: ',(htg_pred['class'] 
                                     - model_pred['class']).numpy().sum())

# Generate predictions on the whole test set
pred_kwargs['steps'] = 10000//pred_kwargs['batch_size']
batch_test  = Batcher(ds_test,  **batch_kwargs)             # Reset batch iterator
model_pred_batch = model.predict(batch_test, **pred_kwargs) # Using a batcher
model_pred_dataset = model.predict(ds_test,**pred_kwargs)   # Using a dataset
print('Test diff:', (model_pred_batch['class'] 
                                   - model_pred_dataset['class']).sum())

# Evaluate the model on the test set
accuracy = model.evaluate((x,y),verbose=False,batch_size=200,return_dict=True)
print(f'Test performance metrics: {accuracy}')

# Close the batchers
batch_train.close()
batch_test.close()

print(f'Notebook runtime: {time.time()-start_time:.1f} seconds')

Accuracy on the full test set should be > ~96%, and runtime < 60 seconds (assuming all cells were run at once). 